# 08 - Final 15-class model comparison

Evaluate the compact baseline and improved MobileNet model on the same held-out signer. Both checkpoints are refit on training plus validation signers; the test signer is excluded from training and model selection.

In [ ]:
import importlib
import sys
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name.lower() == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import src.landmark_image_ensemble as landmark_image_ensemble
import src.robust_landmark_ensemble as robust_landmark_ensemble
import src.ensemble_inference as ensemble_inference
import src.compare_landmark_models as compare_landmark_models

importlib.reload(landmark_image_ensemble)
importlib.reload(robust_landmark_ensemble)
importlib.reload(ensemble_inference)
importlib.reload(compare_landmark_models)

from src.config import load_config, project_path

config = load_config()
output_dir = project_path(config, 'predictions') / 'model_comparison'
improved = project_path(config, 'checkpoints') / 'robust_landmark_ensemble_best.pt'
manifest, _, _ = robust_landmark_ensemble.load_inputs(config)
print('Data split (signer IDs inferred from folder names):')
for split in ('train', 'validation', 'test'):
    rows = manifest['splits'][split]
    print(f"  {split}: {len(rows)} images; signers: {sorted({row['signer_id'] for row in rows})}")
split_counts = pd.DataFrame({split: pd.Series([row['label'] for row in manifest['splits'][split]]).value_counts() for split in ('train', 'validation', 'test')}).fillna(0).astype(int).sort_index()
print('\nImages per class and split:')
print(split_counts.to_string())
comparison = compare_landmark_models.compare_models(config, improved, output_dir)
print('Exploratory test metrics (fractions):')
print(comparison['summary'].to_string(index=False))
print('\nPer-class precision, recall and F1:')
print(comparison['per_class'].to_string(index=False))
print('\nSaved comparison tables, predictions and confusion matrices:', output_dir)
